In [ ]:
import os
import datetime
import colormaps
from pathlib import Path
from functools import partial
from itertools import product
from string import ascii_lowercase
from tqdm import tqdm, trange
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
from matplotlib.colors import (
    LinearSegmentedColormap,
    BoundaryNorm,
    Normalize,
    rgb_to_hsv,
    hsv_to_rgb,
)
from matplotlib.ticker import MaxNLocator
import numpy as np
import pandas as pd
import xarray as xr
import polars as pl
import polars.selectors as cs

os.environ["PATH"] = os.environ["PATH"] + ":/opt/homebrew/bin"

from jetutils.definitions import (
    PRETTIER_VARNAME,
    YEARS,
    UNITS,
    RESULTS,
    FACTORS_UNITS,
    FACTORS,
    DATADIR,
    DUNCANS_REGIONS_NAMES,
    MONTH_NAMES,
    FIGURES,
    RADIUS,
    get_region,
    squarify,
    polars_to_xarray,
    xarray_to_polars,
)
from jetutils.data import standardize, standardize_polars_dtypes, compute_anomalies_pl
from jetutils.geospatial import (
    central_diff,
    haversine_from_dl,
    compute_relative_clim,
    compute_relative_sm,
    compute_relative_std,
    compute_relative_anom,
    create_all_relative_plots,
)
from jetutils.jet_finding import (
    average_jet_categories,
    track_jets,
    pers_from_cross,
    spells_from_cross_catd_simple,
    extract_features,
    find_all_jets,
    jet_position_as_da,
    to_one_large,
)
from jetutils.plots import (
    STYLE_SHEET,
    COLORS,
    COLORS_EXT,
    WERNLI_FLAIR,
    WERNLI_FLAIR_LEVELS,
    Clusterplot,
    plot_interp,
    plot_relative_time,
    gaussian_kde,
)
from jetutils.anyspell import (
    make_daily,
    mask_from_spells_pl,
    subset_around_onset,
    extend_spells,
    get_spells,
)
from jetutils.stats import create_bootstrapped_times, odds_ratio, is_signif_OR, common_OR

%load_ext IPython.extensions.autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

In [ ]:
both_props = {}
both_phat_props = {}
for run in ["historical", "ssp370"]:
    basepath = Path(DATADIR, run)
    # jets = pl.read_parquet(basepath.joinpath("jets.parquet"))
    both_props[run] = pl.read_parquet(basepath.joinpath("props.parquet"))
    both_phat_props[run] = pl.read_parquet(basepath.joinpath("props_full.parquet"))
    cross_catd = pl.read_parquet(basepath.joinpath("cross.parquet"))

In [ ]:
from matplotlib.dates import MonthLocator
from jetutils.data import periodic_rolling_pl
from matplotlib.lines import Line2D
# data_vars: list = ["mean_lat", "mean_s", "mean_theta", "tilt", "wavinessR16", "width", "pers", "int", "int_over_europe"]
data_vars: list = ["mean_lat", "mean_s", "tilt", "wavinessR16", "pers", "int_over_europe"]
nrows: int = 2
ncols: int = 3

plot_kwargs = {"historical": [both_phat_props["historical"], "solid"], "ssp370": [both_phat_props["ssp370"], "dashed"]}

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(ncols * 3.5, nrows * 2.4),
    constrained_layout=True,
    sharex="all",
)
axes = axes.flatten()
jets = both_phat_props["historical"]["jet"].unique().to_numpy()
njets = len(jets)
gb_args = [pl.col("time").dt.ordinal_day().alias("dayofyear"), pl.col("jet")]
aggs = [
    pl.col(col).replace([float("-inf"), float("inf"), float("nan")], None).mean()
    for col in data_vars
]


for name, args in plot_kwargs.items():
    props, ls = args
    gb = props.group_by(gb_args)

    means = gb.agg(aggs).sort("dayofyear", "jet", descending=[False, True])
    means = squarify(means, ["dayofyear", "jet"])
    means = periodic_rolling_pl(means, 15, data_vars)

    x = means["dayofyear"].unique()
    color_order = [2, 1]
    for letter, varname, ax in zip(ascii_lowercase, data_vars, axes.ravel()):
        dji = varname == "double_jet_index"
        factor = FACTORS.get(varname, 1)
        if factor == 1:
            factor_str = ""
        else:
            factor_str = str(int(np.log10(np.abs(factor))))
            factor_str = r"$10^{" + factor_str + r"} \times $"
        ys = means[varname].to_numpy().reshape(366, njets)
        for i in range(njets):
            color = "black" if dji else COLORS[color_order[i]]
            ax.plot(
                x,
                ys[:, i] / factor,
                lw=3,
                color=color,
                zorder=10,
                ls=ls,
            )
            if dji:
                break
        ax.set_title(
            f"{letter}) {PRETTIER_VARNAME.get(varname, varname)} [{factor_str}{UNITS.get(varname, '')}]"
        )
        ax.xaxis.set_major_locator(MonthLocator(range(0, 13, 3)))
        ax.xaxis.set_major_formatter(DateFormatter("%b"))
        ax.set_xlim(min(x), max(x))
        if varname == "mean_lev":
            ax.invert_yaxis()
        # if name == "dobl":
        #     ylim = ax.get_ylim()
        #     wherex = np.isin(x, JJASDOYS)
        #     ax.fill_between(
        #         x, *ylim, where=wherex, alpha=0.1, color="black", zorder=-10
        #     )
        #     ax.set_ylim(ylim)
           
# linedata =  [[0, 1], [0, 1]]
# handles = [
#     Line2D(*linedata, lw=2, ls="solid", color=COLORS[2]),
#     Line2D(*linedata, lw=2, ls="solid", color=COLORS[1]),
#     Line2D(*linedata, lw=2, ls="solid", color="black"),
#     Line2D(*linedata, lw=2, ls="dashed", color="black"),
# ]
# labels = ["STJ", "EDJ", "ctrl", "dobl"]
# axes.ravel()[1].legend(ncol=2, handles=handles, labels=labels, labelspacing=0.2, handletextpad=0.2, columnspacing=0.01, handlelength=1.1).set_zorder(102)
# fig.savefig(f"{FIGURES}/Diabatic/seasonal.pdf")